# Convert NAFNet checkpoint (.pth) sang ONNX — ban LOCAL

Ban local cua `convert_to_onnx.ipynb`, chay truc tiep tren may (khong can Colab,
khong can Google Drive) vi thu muc nay (`image-restoration/`) CHINH LA repo NAFNet
(co san `basicsr/`), va checkpoint `.pth` nam san o `../Data/`.

Chay theo thu tu tu tren xuong. Sau khi co file `.onnx`, dung
`test_onnx_quality_local.ipynb` de kiem tra chat luong.

## Buoc 0. Kiem tra moi truong (GPU, deps)

In [1]:
import torch
print('Torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

Torch: 2.5.1+cu121 | CUDA available: True
GPU: NVIDIA GeForce RTX 3060


## Buoc 1. Import kien truc NAFNet

Khong can `git clone` hay `pip install` repo rieng — notebook nay dang chay
ngay trong thu muc repo NAFNet, chi can dam bao Python dang chay tu thu muc
goc cua repo (`image-restoration/`) de import duoc goi `basicsr`.

In [2]:
import os
import sys

# Dam bao thu muc goc cua repo (chua goi 'basicsr') nam trong sys.path
REPO_ROOT = os.path.abspath('.')
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np
from basicsr.models.archs.NAFNet_arch import NAFNet

print('Import basicsr.models.archs.NAFNet_arch: OK')

Import basicsr.models.archs.NAFNet_arch: OK


## Buoc 2. CONFIG — chinh cac bien o day

- `CHECKPOINT_PATH`: duong dan toi file `.pth` trong `../Data/`.
- `PRESET`: **phai khop chinh xac** voi checkpoint, lay tu `options/train/*/NAFNet-*.yml`:
  - `sidd_width32` / `sidd_width64` — checkpoint SIDD (denoise): `enc=[2,2,4,8] mid=12 dec=[2,2,2,2]`
  - `gopro_width32` / `gopro_width64` — checkpoint GoPro (deblur): `enc=[1,1,1,28] mid=1 dec=[1,1,1,1]`
  - `reds_width64` — checkpoint REDS: giong GoPro, chi khac width=64
- `IMG_H`, `IMG_W`: kich thuoc tile co dinh khi export (boi so cua 16).

**Ghi lai dung 4 gia tri nay** — `test_onnx_quality_local.ipynb` can nhap lai
chinh xac de test dung.

Neu buoc 3 (load checkpoint) bao `size mismatch` -> dang chon sai `PRESET`
so voi `CHECKPOINT_PATH` — sua lai CA HAI dong cho khop nhau.

In [3]:
CHECKPOINT_PATH = '../Data/NAFNet-GoPro-width64.pth'  # <-- sua neu muon test file khac
PRESET = 'gopro_width64'  # <-- phai khop voi CHECKPOINT_PATH o tren

PRESETS = {
    'sidd_width32':  dict(width=32, enc_blk_nums=[2, 2, 4, 8],  middle_blk_num=12, dec_blk_nums=[2, 2, 2, 2]),
    'sidd_width64':  dict(width=64, enc_blk_nums=[2, 2, 4, 8],  middle_blk_num=12, dec_blk_nums=[2, 2, 2, 2]),
    'gopro_width32': dict(width=32, enc_blk_nums=[1, 1, 1, 28], middle_blk_num=1,  dec_blk_nums=[1, 1, 1, 1]),
    'gopro_width64': dict(width=64, enc_blk_nums=[1, 1, 1, 28], middle_blk_num=1,  dec_blk_nums=[1, 1, 1, 1]),
    'reds_width64':  dict(width=64, enc_blk_nums=[1, 1, 1, 28], middle_blk_num=1,  dec_blk_nums=[1, 1, 1, 1]),
}
MODEL_CFG = PRESETS[PRESET]

IMG_H, IMG_W = 256, 256

# Tat ca file ket qua (.onnx) luu vao thu muc Result/ chung (ngang hang voi Data/)
ONNX_OUTPUT_DIR = '../Result/onnx'
ONNX_OUTPUT_PATH = f'{ONNX_OUTPUT_DIR}/nafnet_{PRESET}_{IMG_H}x{IMG_W}.onnx'
ONNX_SIMPLIFIED_PATH = ONNX_OUTPUT_PATH.replace('.onnx', '_sim.onnx')

size_mb = os.path.getsize(CHECKPOINT_PATH) / 1024 / 1024
print(f'Checkpoint: {CHECKPOINT_PATH} ({size_mb:.2f} MB)')
print('Preset:', PRESET, MODEL_CFG)
print('ONNX se luu tai:', ONNX_SIMPLIFIED_PATH)

Checkpoint: ../Data/NAFNet-GoPro-width64.pth (259.19 MB)
Preset: gopro_width64 {'width': 64, 'enc_blk_nums': [1, 1, 1, 28], 'middle_blk_num': 1, 'dec_blk_nums': [1, 1, 1, 1]}
ONNX se luu tai: ../Result/onnx/nafnet_gopro_width64_256x256_sim.onnx


## Buoc 3. Build model va load checkpoint

In [4]:
model = NAFNet(img_channel=3, **MODEL_CFG)

ckpt = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
state_dict = ckpt.get('params', ckpt) if isinstance(ckpt, dict) else ckpt

missing, unexpected = model.load_state_dict(state_dict, strict=False)
assert len(missing) == 0 and len(unexpected) == 0, (
    f'State dict khong khop kien truc (missing={missing}, unexpected={unexpected}) '
    f'— kiem tra lai PRESET co dung voi CHECKPOINT_PATH khong.'
)
model.eval()

# Kiem tra nhanh trong so co "song" khong (phat hien checkpoint hong/rong)
for name in ['intro.weight', 'ending.weight']:
    p = dict(model.named_parameters())[name]
    print(f'{name}: mean={p.data.mean().item():.6f} std={p.data.std().item():.6f}')

print('Load checkpoint thanh cong.')

intro.weight: mean=0.000180 std=0.195915
ending.weight: mean=0.000075 std=0.004493
Load checkpoint thanh cong.


## Buoc 4. Export sang ONNX

In [5]:
os.makedirs(ONNX_OUTPUT_DIR, exist_ok=True)

dummy_input = torch.randn(1, 3, IMG_H, IMG_W)

with torch.no_grad():
    torch.onnx.export(
        model,
        dummy_input,
        ONNX_OUTPUT_PATH,
        input_names=['input'],
        output_names=['output'],
        opset_version=17,
        do_constant_folding=True,
        dynamo=False,  # exporter kieu cu (TorchScript-based), on dinh hon voi kien truc custom
    )

print('Da export ONNX:', ONNX_OUTPUT_PATH)

Da export ONNX: ../Result/onnx/nafnet_gopro_width64_256x256.onnx


## Buoc 5. Simplify ONNX graph

In [6]:
import onnx
from onnxsim import simplify

# Dung Python API thay vi CLI 'python -m onnxsim' de tranh bug crash trong
# buoc in bang thong ke (sympy.factor treo voi so lon). Ket qua simplify khong doi.
model_onnx = onnx.load(ONNX_OUTPUT_PATH)
model_simplified, check = simplify(model_onnx)
assert check, 'Simplified ONNX model khong hop le'

onnx.save(model_simplified, ONNX_SIMPLIFIED_PATH)
print('Da simplify:', ONNX_SIMPLIFIED_PATH)

Da simplify: ../Result/onnx/nafnet_gopro_width64_256x256_sim.onnx


## Buoc 6. Verify: so sanh output ONNX vs PyTorch

In [7]:
import ctypes
import glob
import nvidia

# onnxruntime-gpu can dlopen libcublas/libcudnn theo dung SONAME (vd libcublasLt.so.12).
# PyTorch da mang san cac thu vien nay trong .venv nhung khong nam trong LD_LIBRARY_PATH cua
# tien trinh dang chay (bien nay chi duoc dynamic linker doc luc KHOI DONG tien trinh, sua
# os.environ luc runtime khong co tac dung) — nen preload thu cong bang ctypes (RTLD_GLOBAL).
# QUAN TRONG: phai LOAI TRU libnvblas.so — no "hook" lai BLAS toan cuc va gay segfault khi
# preload chung voi cuBLAS cua PyTorch (da kiem chung thuc te).
_nvidia_root = os.path.dirname(nvidia.__file__)
for _so in glob.glob(f'{_nvidia_root}/*/lib/*.so*'):
    if 'nvblas' in _so:
        continue
    try:
        ctypes.CDLL(_so, mode=ctypes.RTLD_GLOBAL)
    except OSError:
        pass

# QUAN TRONG: GPU dong Ampere (RTX 30xx tro len) mac dinh dung TF32 (giam do chinh xac de
# tang toc). Voi model 28 NAFBlock lien tiep, sai so TF32 tich luy rat lon (mean diff ~0.1,
# max ~0.66 — da kiem chung thuc te) — phai tat TF32 (torch + onnxruntime provider option
# 'use_tf32': 0) de ket qua GPU khop voi CPU/mobile that.
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

import onnxruntime as ort

USE_GPU_FOR_VERIFY = torch.cuda.is_available() and 'CUDAExecutionProvider' in ort.get_available_providers()
if USE_GPU_FOR_VERIFY:
    providers = [('CUDAExecutionProvider', {'use_tf32': 0}), 'CPUExecutionProvider']
else:
    providers = ['CPUExecutionProvider']

sess = ort.InferenceSession(ONNX_SIMPLIFIED_PATH, providers=providers)
print('ONNX Runtime providers:', sess.get_providers())

device = 'cuda' if USE_GPU_FOR_VERIFY else 'cpu'
model_verify = model.to(device)

test_input = torch.randn(1, 3, IMG_H, IMG_W)
with torch.no_grad():
    out_torch = model_verify(test_input.to(device)).cpu().numpy()
out_onnx = sess.run(None, {'input': test_input.numpy()})[0]

diff = np.abs(out_torch - out_onnx)
max_diff, mean_diff = diff.max(), diff.mean()
print('Max abs diff:', max_diff, '| Mean abs diff:', mean_diff)

# mean_diff la chi so chinh (dai dien cho toan anh). max_diff de canh bao neu
# qua bat thuong; model cang lon (width64) cang tich luy sai so lam tron nhieu hon.
assert mean_diff < 5e-3, 'Sai so TRUNG BINH qua lon — kiem tra lai export/simplify'
assert max_diff < 0.15, 'Sai so LON NHAT qua bat thuong — co the co loi export'
print('OK: output ONNX khop voi PyTorch.')

ONNX Runtime providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']


Max abs diff: 0.0012979507 | Mean abs diff: 0.0001798172
OK: output ONNX khop voi PyTorch.


## Buoc 7. Hoan tat

File `.onnx` da duoc luu vao thu muc `../Result/onnx/`:
- `ONNX_OUTPUT_PATH` (ban goc)
- `ONNX_SIMPLIFIED_PATH` (da simplify — nen dung file nay cho mobile)

**Buoc tiep theo:** mo `test_onnx_quality_local.ipynb`, nhap lai dung
`CHECKPOINT_PATH`, `PRESET`, `IMG_H`, `IMG_W` o phan CONFIG cua file do
(no tu suy ra duong dan `.onnx` tu cac gia tri nay).